# 08. 자원최적화 및 현장활용: 월간 생산 재배치와 비용 최소화

하루 생산량을 고정하지 않고 8월 안에서 생산 날짜와 시간을 바꿉니다. 주간에서 야간으로, 야간에서 주간으로도 옮길 수 있습니다. 분석기간 전체 생산량은 유지하면서 기본요금, 사용요금, 작업 인건비를 계산합니다.

**실행 범위:** 2021년 8월 1~31일. 04~06 및 09~15의 기존 모델 결과는 사용하지 않습니다. 기존 17의 팀원 예측 패키지만 재사용합니다. 첫 실행에서 주별 LightGBM을 다시 학습하고 많은 이동 후보를 비교하므로 시간이 걸립니다.

**결과 의미:** 명시한 가정 아래 원 단위로 계산한 모델 기반 시뮬레이션입니다. 실제 절감액이나 모든 가능한 계획 중 절대적인 최저 비용을 뜻하지 않습니다. 후보를 반복 비교해 찾은 가장 낮은 비용의 계획입니다. 전체 월을 알고 탐색하는 사후 계획 비교이며, 납기·제품별 작업순서·설비별 생산능력은 데이터에 없어 반영하지 못했습니다. 각 날짜의 과거 입력은 관측 이력을 고정하므로 앞 날짜의 변경이 뒤 날짜 입력에 미치는 영향도 재현하지 않습니다.


## 1. 비용 기준과 출처

|항목|이번 계산 기준|상태|
|---|---|---|
|전력 단위|원본은 15분 평균수요전력 kW로 해석, 각 칸에 0.25시간을 곱해 kWh 계산|가이드북 PDF 7~9쪽의 최대수요전력 설명에 근거한 해석. 원자료 단위 확정 시 재확인|
|사용요금|원본의 `전기요금(계절)`을 원/kWh로 가정|공장 실제 계약 단가로 검증되지 않은 가정|
|기본요금 단가|8,320원/kW·월|과거 산업용 고압A 선택II 참고 단가. 해당 공장의 2021년 계약은 확인되지 않음|
|시간당 임금|8,720원|2021년 최저임금, 제조업 평균임금은 아님|
|작업 인원|10명 기준, 5·10·20명 민감도|작업조 가정이며 통계청 평균 인원이 아님|
|작업 인건비|원본 주야간 인건비 비율 × 시급 × 작업 인원 × 생산이 배치된 시간|생산이 완전히 빠진 시간은 비용에서 제외하고 새 작업시간은 추가|

- [고용노동부 2021년 최저임금 고시](https://www.moel.go.kr/news/enews/report/enewsView.do?news_seq=11245)
- [에너지경제연구원 과거 산업용 요금 참고표](https://www.keei.re.kr/boardDownload.es?bid=0001&list_no=82003&seq=1)
- [통계청 2021년 광업제조업조사](https://mods.go.kr/board.es?act=view&bid=232&list_no=422261&mid=a10301090100): 10인 이상 사업체 전체 통계만으로 이 공장 작업조 인원을 결정할 수 없어 직접 대입하지 않음.
- 데이터 가이드북 9쪽: `공장인원`은 생산력, `인건비`는 주야간 비율. 실제 사람 수로 계산하지 않음.

**기본요금 2가지:** `fixed`는 기존 예측 월 최대전력에 기본요금이 고정된 경우입니다. `monthly_peak_assumption`은 변경 후 월 최대전력에 따라 기본요금이 달라진다고 가정한 경우입니다. 최적화는 기본요금 절감 가능성을 포함하기 위해 `monthly_peak_assumption`으로 수행하고, 같은 계획을 `fixed`로도 다시 계산합니다.

현재 8월의 계절 단가는 시간과 날짜에 따라 달라지지 않습니다. 따라서 시간 이동 자체로 사용요금 단가가 낮아지지는 않습니다. 사용요금 변화는 생산 재배치 후 모델이 예측한 총 전력사용량 변화에서 발생합니다.


In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), Path.cwd().parent]
    if (p / 'results/team_submission_runtime/outputs').exists()
    and (p / 'real_final/08_resource_cost_optimization.py').exists()
)

SETTINGS = {
    'month': '2021-08',
    'hourly_wage': 8720.0,
    'staff': 10,
    'staff_sensitivity': [5, 10, 20],
    'basic_won_per_kw': 8320.0,
    'max_iterations_by_objective': {'peak': 50, 'electricity': 30, 'total': 40},
    'source_pool_size': 8,
    'destination_pool_size': 10,
    'electricity_source_pool_size': 12,
    'electricity_destination_pool_size': 12,
    'peak_source_pool_size': 16,
    'peak_destination_pool_size': 16,
    'peak_destination_scan_size': 96,
    'peak_max_destination_legs': 8,
    'peak_max_source_group': 3,
    'peak_move_fractions': [1.0, 0.75, 0.5, 0.35, 0.25, 0.15, 0.1],
    'peak_group_fractions': [0.5, 0.35, 0.25, 0.15],
    'move_fractions': [1.0, 0.5, 0.25],
    'minimum_improvement_won': 1.0,
    'minimum_peak_improvement_kw': 0.001,
    'active_epsilon': 1e-6,
    'safety_tolerance': 1e-5,
    'unit_interpretation': '15-minute average kW; kWh = kW * 0.25',
    'tariff_status': 'dataset seasonal rate assumed won/kWh; basic rate historical reference, not verified 2021 contract',
    'labor_status': 'assumed crew; 2021 minimum wage; no payroll or statutory overtime claim',
    'interpretation': 'retrospective monthly conditional model simulation; not observed causal savings',
}
print(SETTINGS)


## 2. 탐색 방법과 비교안

|안|선택 기준|
|---|---|
|baseline|기존 생산계획|
|peak|월 최대전력 최소화, 같은 최대전력이면 전기요금과 총비용이 낮은 계획 선택|
|electricity|기본요금 + 사용요금 최소화|
|total|기본요금 + 사용요금 + 작업 인건비 최소화|

기존 방식의 같은 날 ±6시간, 06~21시, 하루 생산량 2% 제한을 제거했습니다. 8월의 모든 날짜와 0~23시를 이동 후보로 사용합니다. 하루 생산량은 달라질 수 있지만 8월 전체 생산량은 정확히 유지합니다.

한 시간에 넣을 수 있는 생산량은 실제 설비 한도가 없으므로 과거 해당 시간 생산량의 상위 95%를 참고 한도로 사용합니다. 전기요금안과 총비용안은 기존 시간 생산량의 100%, 50%, 25% 이동 후보를 비교합니다.

피크 저감안은 현재 수치예측 전력이 가장 큰 시간을 우선 이동 후보에 넣고, 월 최대전력이 실제로 낮아지는 이동만 선택합니다. 한 피크 시간의 생산량을 최대 8개의 저부하 시간으로 나누어 옮기며, 상위 2~3개 피크 시간을 한꺼번에 조정하는 묶음 이동도 비교합니다. 도착지는 예측전력이 낮은 시간뿐 아니라 생산 여유가 큰 시간도 포함합니다. 하나의 피크를 낮춘 뒤 다음으로 높은 날짜의 피크를 이어서 낮추며, 같은 최대전력이면 전기요금과 총비용이 낮은 계획을 선택합니다.

전기요금안은 기본요금과 사용요금의 합계를 최소화하고, 총비용안은 인건비 비율이 높은 시간을 우선 줄이는 기존 방식을 유지합니다.

피크 저감안은 최대 50회, 전기요금안은 최대 30회, 총비용안은 최대 40회 반복합니다. 피크 저감안은 여러 시간의 생산을 묶어서 평가하므로 이전 실행보다 시간이 더 걸릴 수 있습니다. 모든 가능한 계획을 수학적으로 전부 계산하는 방식은 아니므로 결과는 현재 데이터와 생산한도 안에서 폭넓게 탐색한 최선의 개선안입니다.

월 최대전력, LightGBM Q1 경보칸 수, 기준 176 초과 정도는 기존 계획보다 늘지 않도록 제한합니다. 최종 경보 모델은 앙상블이지만, 변경된 생산계획에 대해 GRU를 다시 계산하는 기능이 없어 이 셀의 안전 점검에는 재계산 가능한 LightGBM Q1을 사용합니다. 실제 적용 전에는 최종 앙상블과 164.5 주의 단계로 다시 확인해야 합니다.

인건비는 생산이 배치된 시간만 계산합니다. 생산을 다른 시간으로 모두 옮겨 기존 작업시간이 비면 그 시간의 비용은 줄고, 비가동 시간에 생산을 새로 배치하면 해당 시간의 비용이 추가됩니다.


In [ ]:
module_path = ROOT / 'real_final/08_resource_cost_optimization.py'
spec = importlib.util.spec_from_file_location('chapter4_cost', module_path)
chapter4 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(chapter4)
# 이 셀에서 모델 재학습과 계획 탐색이 실행됩니다.
comparison = chapter4.run(ROOT, SETTINGS)


## 3. 4장에 사용할 표와 현장 조치

`monthly_peak_assumption`은 월 최대전력이 낮아지면 기본요금도 낮아진다는 가정이고, `fixed`는 계약전력이 이미 고정되어 기본요금이 줄지 않는 경우입니다. 두 표를 함께 제시하면 기본요금 가정에 따른 차이를 설명할 수 있습니다.

현장에서는 다음 순서로 결과를 사용합니다.

1. `comparison_summary.csv`에서 기본요금, 사용요금, 인건비와 총비용 절감액을 확인합니다.
2. `accepted_moves.csv`에서 어느 날짜·시간의 생산을 어디로 옮겼는지 확인합니다.
3. `daily_production_comparison.csv`에서 날짜별 생산량 증감과 납기 가능 여부를 확인합니다.
4. `hourly_plans.csv`에서 시간별 생산량과 참고 생산한도를 확인합니다.
5. 실제 적용 전 설비 용량, 제품별 순서, 교대조, 납기를 확인하고 최종 앙상블 경보와 주의 단계를 다시 계산합니다.

모델이 제안한 이동을 바로 적용하는 것이 아니라 현장 조건을 통과한 이동만 생산계획에 반영합니다.


In [ ]:
columns = [
    'basic_mode', 'objective', 'production', 'moved_production',
    'peak_kw', 'peak_kw_saving', 'kwh', 'kwh_saving',
    'basic_won', 'basic_won_saving', 'usage_won', 'usage_won_saving',
    'electricity_won', 'labor_won', 'labor_won_saving',
    'total_won', 'total_won_saving', 'saving_percent',
    'alarm_cells', 'changed_days', 'stop_reason',
]
display(comparison[columns].round(2))

result_dir = ROOT / 'results/resource_cost_optimization_chapter4'
print('결과:', result_dir)
print('accepted_moves.csv: 날짜와 시간을 넘겨 이동한 생산량')
print('daily_production_comparison.csv: 날짜별 기존·개선 생산량과 증감')
print('hourly_plans.csv: 시간별 기존·개선 생산량, 주야간, 인건비 비율, 참고 생산한도')
print('staff_wage_sensitivity.csv: 같은 계획을 인원·임금별로 다시 계산한 결과이며 재최적화 결과는 아닙니다.')
